# Dafne + MedSAM Thigh Segmentation — Water (Lambda)\n\nRuns the Dafne Thigh model slice-by-slice on **Dixon WATER** stacks, then refines each\nmuscle mask with MedSAM using the Dafne bounding box as the prompt.\n\nMedSAM embedding is computed **once per slice** and reused for all muscles.\nDafne (TensorFlow) runs on GPU; MedSAM stays on CPU to avoid VRAM conflicts.\n\nFiles are processed in **reverse alphabetical order** (P004 first, HV001 last)\nso that already-completed files are skipped and new results appear first.\n\n## Before running — upload to Lambda\n\n```bash\n# water images\nrsync -avz -e \"ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no\" \\\n  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/myosegmenTUM \\\n  your machine9:~/\n\n# Dafne thigh model\nrsync -avz -e \"ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no\" \\\n  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne/extras/model/Thigh_1774532147.model \\\n  your machine9:~/Thigh_1774532147.model\n\n# MedSAM checkpoint\nrsync -avz -e \"ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no\" \\\n  \"/tmp/docker-desktop-root/run/desktop/mnt/host/c/Users/docto/AppData/Local/Dafne-imaging/Dafne/models/medsam_vit_b.pth\" \\\n  your machine9:~/medsam_vit_b.pth\n```\n\n## Download results when done\n\n```bash\nrsync -avz -e \"ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no\" \\\n  your machine9:~/dafne_medsam_results_water/ \\\n  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne_medsam_results_water/\n```\n\n**Terminate the instance when done.**

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'dafne-dl', 'SimpleITK', 'scikit-image'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/facebookresearch/segment-anything.git'])

In [ ]:
import glob
import os
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F
from skimage import transform
from dafne_dl import DynamicDLModel
from segment_anything import sam_model_registry

In [ ]:
# inlined from dafne.utils.sam_mask_refine
def enlarge_bounding_box(mask, margin=5):
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    H, W = mask.shape
    return np.array([
        max(0, cmin - margin), max(0, rmin - margin),
        min(W - 1, cmax + margin), min(H - 1, rmax + margin),
    ], dtype=float)


def medsam_inference(medsam_model, img_embed, box_1024, H, W):
    box_torch = torch.as_tensor(box_1024, dtype=torch.float, device=img_embed.device)
    if box_torch.ndim == 2:
        box_torch = box_torch[:, None, :]
    sparse_embeddings, dense_embeddings = medsam_model.prompt_encoder(
        points=None, boxes=box_torch, masks=None
    )
    low_res_logits, _ = medsam_model.mask_decoder(
        image_embeddings=img_embed,
        image_pe=medsam_model.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=False,
    )
    low_res_pred = F.interpolate(
        torch.sigmoid(low_res_logits), size=(H, W),
        mode='bilinear', align_corners=False
    )
    return (low_res_pred.squeeze().detach().cpu().numpy() > 0.5).astype(np.uint8)

In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

SAM_DEVICE  = 'cpu'   # keep MedSAM on CPU; TF/Dafne uses GPU

MODEL_PATH  = os.path.expanduser('~/Thigh_1774532147.model')
SAM_CKPT    = os.path.expanduser('~/medsam_vit_b.pth')
IMAGE_GLOB  = os.path.expanduser('~/myosegmenTUM/*/ImageData/*_WATER/*_WATER_stack*.nii')
OUTPUT_DIR  = os.path.expanduser('~/dafne_medsam_results_water')

os.makedirs(OUTPUT_DIR, exist_ok=True)

for path, label in [(MODEL_PATH, 'Dafne model'), (SAM_CKPT, 'MedSAM checkpoint')]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'{label} not found: {path} — upload it first.')

print('SAM device :', SAM_DEVICE)
print('Model      :', MODEL_PATH)
print('Output     :', OUTPUT_DIR)

In [ ]:
dafne_model = DynamicDLModel.Load(open(MODEL_PATH, 'rb'))
print('Dafne model loaded:', MODEL_PATH)

In [ ]:
sam_model = sam_model_registry['vit_b'](checkpoint=SAM_CKPT)
sam_model.to(device=SAM_DEVICE)
sam_model.eval()
print('MedSAM loaded on', SAM_DEVICE)

In [ ]:
# reverse=True -> P004 first, HV001 last
image_files = sorted(glob.glob(IMAGE_GLOB), reverse=True)
print(f'Found {len(image_files)} water stacks (reverse order):')
for p in image_files:
    print(' ', p)

In [ ]:
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_dafne_medsam.npz')

    if os.path.exists(out_path):
        print(f'Skipping (already done): {stem}')
        continue

    print(f'\nProcessing: {nii_path}')
    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)
    spacing   = img_sitk.GetSpacing()
    resolution = [spacing[0], spacing[1]]
    H, W = img_array.shape[1], img_array.shape[2]
    print(f'  Shape: {img_array.shape}  Resolution: {resolution}')

    all_masks = {}

    for slice_idx in range(img_array.shape[0]):
        slice_2d = img_array[slice_idx]

        # Dafne segmentation
        dafne_out = dafne_model({
            'image': slice_2d,
            'resolution': resolution,
            'split_laterality': True,
            'classification': 'Thigh',
        })

        # MedSAM embedding — once per slice, reused for all muscles
        img_norm   = slice_2d * 255.0 / (slice_2d.max() + 1e-8)
        img_3c     = np.repeat(img_norm[:, :, None], 3, axis=-1)
        img_1024   = transform.resize(
            img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
        ).astype(np.uint8)
        img_1024   = (img_1024 - img_1024.min()) / np.clip(
            img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
        )
        img_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(SAM_DEVICE)
        with torch.no_grad():
            image_embedding = sam_model.image_encoder(img_tensor)
        del img_tensor

        # refine each muscle mask with MedSAM bounding-box prompt
        for muscle_name, mask in dafne_out.items():
            mask_arr = np.asarray(mask, dtype=np.uint8)

            if mask_arr.any():
                bbox     = enlarge_bounding_box(mask_arr)
                box_1024 = bbox / np.array([W, H, W, H]) * 1024
                box_1024 = box_1024[None, None, :]
                refined  = medsam_inference(sam_model, image_embedding, box_1024, H, W)
            else:
                refined = mask_arr

            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = refined.astype(np.uint8)

        del image_embedding

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f'  slice {slice_idx + 1}/{img_array.shape[0]} done')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved -> {out_path}')
    print(f'  Muscles: {list(all_masks.keys())}')

print('\nAll done.')

In [ ]:
# sanity check — reload one result
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Total output files: {len(results)}')
if results:
    sample = np.load(results[0])
    print('Sample file:', results[0])
    for name in sample.files:
        arr = sample[name]
        print(f'  {name}: shape={arr.shape}  positive voxels={arr.sum()}')